# Notebook 04b — Temperature Multi-Site Analysis
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/polar-bear-after-lunch/AareML/blob/main/notebooks/04b_multisite_temperature.ipynb)

**Key insight:** Temperature is measured at all 86 CAMELS-CH-Chem gauges (vs only 16 for DO).
Using temperature as the primary target transforms AareML from a 1-gauge study to a genuine
80+ gauge multi-site analysis across all of Switzerland.

**This notebook:**
1. Identifies all gauges with sufficient temperature data (≥50% coverage)
2. Trains a single-site temperature LSTM on the focus gauge (2473)
3. Evaluates zero-shot transfer to all valid gauges
4. Performs per-gauge retraining across the full network
5. Correlates prediction skill with catchment attributes
6. Maps RMSE across Switzerland (if geopandas available)

In [1]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    import os
    from pathlib import Path
    if not str(Path.cwd()).endswith('AareML'):
        if not Path('AareML').exists():
            os.system('git clone https://github.com/polar-bear-after-lunch/AareML.git')
        os.chdir('AareML')
    os.system('pip install -q -r requirements.txt')
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DATA = Path('/content/drive/MyDrive/AareML_data')
    LOCAL_DATA = Path('data')
    LOCAL_DATA.mkdir(exist_ok=True)
    DRIVE_CAMELS = DRIVE_DATA / 'camels-ch-chem'
    LOCAL_CAMELS = LOCAL_DATA / 'camels-ch-chem'
    if DRIVE_CAMELS.exists() and not LOCAL_CAMELS.exists():
        os.system(f'ln -s {DRIVE_CAMELS} {LOCAL_CAMELS}')
    print(f'Setup complete. cwd: {os.getcwd()}')

In [2]:
import os, multiprocessing
N_CPU = multiprocessing.cpu_count()
N_THREADS = min(N_CPU, 6)
os.environ['OMP_NUM_THREADS']      = str(N_THREADS)
os.environ['MKL_NUM_THREADS']      = str(N_THREADS)
os.environ['OPENBLAS_NUM_THREADS'] = str(N_THREADS)
print(f'CPU cores: {N_CPU} | Using {N_THREADS} threads')

CPU cores: 128 | Using 6 threads


In [3]:
import sys; sys.path.insert(0, '..')
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

from src.config import (
    LOOKBACK, HORIZON, SEED, TRAIN_END, VAL_END, FOCUS_GAUGE,
    FEATURES_TEMP, TARGETS_TEMP, N_FEAT_TEMP, N_TGT_TEMP, TARGET_LABELS_TEMP,
    TEMP_MIN_COVERAGE,
)
from src.data import load_gauge, preprocess, train_val_test_split, make_windows, load_metadata
from src.model import (
    RiverDataset, Seq2SeqLSTM, train_model, predict, get_y_true,
    save_checkpoint, load_checkpoint, reconstruct_scalers,
)
from src.metrics import metrics_table, mean_rmse, block_bootstrap_ci

import random
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

_repo_root  = Path('.') if Path('figures').exists() else Path('..')
FIGURES_DIR = _repo_root / 'figures'
RESULTS_DIR = _repo_root / 'results'
FIGURES_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

sns.set_style('whitegrid')
plt.rcParams.update({'figure.dpi': 130})
print(f'Features: {FEATURES_TEMP}')
print(f'Targets:  {TARGETS_TEMP}')

Features: ['temp_sensor', 'pH_sensor', 'ec_sensor']
Targets:  ['temp_sensor']


In [4]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_WORKERS = 4 if DEVICE.type == 'cuda' else 0
PIN_MEMORY  = DEVICE.type == 'cuda'
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# LOCAL_TEST: True = fast CPU verification, False = full run
LOCAL_TEST = DEVICE.type == 'cpu'
if LOCAL_TEST:
    print('LOCAL_TEST mode: reduced epochs/trials for quick verification')


Device: cuda


GPU: NVIDIA GeForce RTX 4090
VRAM: 23.8 GB


## 1. Gauge Discovery — Temperature Coverage

Unlike DO, temperature is observed at virtually all 86 CAMELS-CH-Chem gauges.
This section identifies all gauges with ≥50% temperature coverage and ≥50 valid test windows.

In [5]:
meta = load_metadata()
all_gauge_ids = meta['gauge_id'].astype(str).tolist() if 'gauge_id' in meta.columns else meta.index.astype(str).tolist()
print(f'Total gauges in metadata: {len(all_gauge_ids)}')

temp_gauges = []
coverage_dict = {}

print('Scanning all gauges for temperature coverage...')
for gid in tqdm(all_gauge_ids, desc='Gauges'):
    try:
        raw  = load_gauge(gid)
        data = preprocess(raw)
        cov  = data['temp_sensor'].notna().mean()
        coverage_dict[gid] = float(cov)
        if cov >= TEMP_MIN_COVERAGE:
            # Check enough test windows
            train_g, _, test = train_val_test_split(data)
            means = (pd.concat([train_g[FEATURES_TEMP].mean(), train_g[TARGETS_TEMP].mean()])
                     .groupby(level=0).first())
            try:
                X_te, y_te, _ = make_windows(test, means,
                                             features=FEATURES_TEMP, targets=TARGETS_TEMP)
                if len(X_te) >= 50:
                    temp_gauges.append(gid)
            except ValueError:
                pass
    except Exception:
        pass

temp_gauges = [g for g in temp_gauges if g != FOCUS_GAUGE]

print(f'\nGauges with ≥{TEMP_MIN_COVERAGE:.0%} temp coverage: {sum(1 for v in coverage_dict.values() if v >= TEMP_MIN_COVERAGE)}')
print(f'Gauges with ≥50 valid test windows: {len(temp_gauges)}')
print(f'Gauge IDs: {temp_gauges}')

# Coverage distribution
cov_series = pd.Series(coverage_dict)
print(f'\nTemperature coverage across all gauges:')
print(f'  Mean: {cov_series.mean():.1%}')
print(f'  Median: {cov_series.median():.1%}')
print(f'  Min: {cov_series.min():.1%}')
print(f'  Gauges with >90% coverage: {(cov_series > 0.9).sum()}')

Total gauges in metadata: 115
Scanning all gauges for temperature coverage...


Gauges:   0%|          | 0/115 [00:00<?, ?it/s]

[data] load_gauge 2009: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.0, 'pH_sensor': 0.022, 'ec_sensor': 0.011, 'O2C_sensor': 0.016}
[data] split: train=12418, val=731, test=1461
[data] make_windows: 1427 windows, X=(1427, 21, 3), y=(1427, 14, 1), date range 2017-01-22 → 2020-12-18
[data] load_gauge 2011: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.0, 'pH_sensor': 1.0, 'ec_sensor': 1.0, 'O2C_sensor': 1.0}
[data] split: train=12418, val=731, test=1461


[data] load_gauge 2016: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.0, 'pH_sensor': 0.054, 'ec_sensor': 0.05, 'O2C_sensor': 0.063}
[data] split: train=12418, val=731, test=1461
[data] make_windows: 1427 windows, X=(1427, 21, 3), y=(1427, 14, 1), date range 2017-01-22 → 2020-12-18
[data] load_gauge 2018: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.0, 'pH_sensor': 0.576, 'ec_sensor': 0.564, 'O2C_sensor': 0.568}
[data] split: train=12418, val=731, test=1461
[data] make_windows: 1427 windows, X=(1427, 21, 3), y=(1427, 14, 1), date range 2017-01-22 → 2020-12-18
[data] load_gauge 2019: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.0, 'pH_sensor': 1.0, 'ec_sensor': 1.0, 'O2C_s

[data] load_gauge 2029: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.0, 'pH_sensor': 1.0, 'ec_sensor': 1.0, 'O2C_sensor': 1.0}
[data] split: train=12418, val=731, test=1461
[data] load_gauge 2030: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.0, 'pH_sensor': 1.0, 'ec_sensor': 1.0, 'O2C_sensor': 1.0}
[data] split: train=12418, val=731, test=1461
[data] load_gauge 2033: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.525, 'pH_sensor': 1.0, 'ec_sensor': 1.0, 'O2C_sensor': 1.0}
[data] load_gauge 2034: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.0, 'pH_sensor': 1.0, 'ec_sen

...

[data] load_gauge 2432: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.525, 'pH_sensor': 1.0, 'ec_sensor': 1.0, 'O2C_sensor': 1.0}
[data] load_gauge 2433: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.745, 'pH_sensor': 1.0, 'ec_sensor': 1.0, 'O2C_sensor': 1.0}
[data] load_gauge 2434: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.803, 'pH_sensor': 1.0, 'ec_sensor': 1.0, 'O2C_sensor': 1.0}
[data] load_gauge 2457: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.0, 'pH_sensor': 1.0, 'ec_sensor': 1.0, 'O2C_sensor': 1.0}
[data] split: train=12418, val=731, test=1461


[data] load_gauge 2462: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.0, 'pH_sensor': 0.466, 'ec_sensor': 0.459, 'O2C_sensor': 0.543}
[data] split: train=12418, val=731, test=1461
[data] make_windows: 1427 windows, X=(1427, 21, 3), y=(1427, 14, 1), date range 2017-01-22 → 2020-12-18
[data] load_gauge 2467: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.0, 'pH_sensor': 1.0, 'ec_sensor': 0.808, 'O2C_sensor': 1.0}
[data] split: train=12418, val=731, test=1461
[data] load_gauge 2473: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.0, 'pH_sensor': 0.01, 'ec_sensor': 0.007, 'O2C_sensor': 0.012}
[data] split: train=12418, val=731, test=1461
[data] make_windows: 1427 windows, X=(1427,

[data] load_gauge 2481: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.035, 'pH_sensor': 1.0, 'ec_sensor': 1.0, 'O2C_sensor': 1.0}
[data] split: train=12418, val=731, test=1461
[data] load_gauge 2485: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.525, 'pH_sensor': 1.0, 'ec_sensor': 1.0, 'O2C_sensor': 1.0}
[data] load_gauge 2493: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.767, 'pH_sensor': 1.0, 'ec_sensor': 1.0, 'O2C_sensor': 1.0}
[data] load_gauge 2500: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.192, 'pH_sensor': 1.0, 'ec_sensor': 1.0, 'O2C_sensor': 1.0}
[data] spl

[data] load_gauge 2606: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.45, 'pH_sensor': 1.0, 'ec_sensor': 1.0, 'O2C_sensor': 1.0}
[data] split: train=12418, val=731, test=1461
[data] load_gauge 2608: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.55, 'pH_sensor': 1.0, 'ec_sensor': 1.0, 'O2C_sensor': 1.0}
[data] load_gauge 2609: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.575, 'pH_sensor': 1.0, 'ec_sensor': 1.0, 'O2C_sensor': 1.0}
[data] load_gauge 2612: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.568, 'pH_sensor': 1.0, 'ec_sensor': 1.0, 'O2C_sensor': 1.0}
[data] load_

[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.35, 'pH_sensor': 0.023, 'ec_sensor': 0.008, 'O2C_sensor': 0.016}
[data] split: train=12418, val=731, test=1461
[data] make_windows: 1427 windows, X=(1427, 21, 3), y=(1427, 14, 1), date range 2017-01-22 → 2020-12-18
[data] load_gauge 2615: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.885, 'pH_sensor': 1.0, 'ec_sensor': 1.0, 'O2C_sensor': 1.0}
[data] load_gauge 2617: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.55, 'pH_sensor': 1.0, 'ec_sensor': 1.0, 'O2C_sensor': 1.0}
[data] load_gauge 2623: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.55, 'pH_sensor': 1.0, 'ec_sensor': 1.0, 'O2C_sensor': 1.0}
[data] load_gauge 2634: 1

## 2. Single-Site Temperature LSTM (Focus Gauge 2473)

Train a Seq2Seq LSTM on gauge 2473 with temperature as the sole target.
Features: [temp_sensor, pH_sensor, ec_sensor] — excluding DO to avoid
target leakage (temperature predicts temperature, but DO and temperature
are correlated so including DO as a feature would inflate performance).

In [6]:
raw_focus   = load_gauge(FOCUS_GAUGE)
data_focus  = preprocess(raw_focus)
train_focus, val_focus, test_focus = train_val_test_split(data_focus)

# Training means for imputation (no leakage)
train_means_focus = (
    pd.concat([train_focus[FEATURES_TEMP].mean(), train_focus[TARGETS_TEMP].mean()])
    .groupby(level=0).first()
)

# Build windows
X_train, y_train, _ = make_windows(train_focus, train_means_focus,
                                    features=FEATURES_TEMP, targets=TARGETS_TEMP)
X_val,   y_val,   _ = make_windows(val_focus,   train_means_focus,
                                    features=FEATURES_TEMP, targets=TARGETS_TEMP)
X_test,  y_test,  d_test = make_windows(test_focus, train_means_focus,
                                         features=FEATURES_TEMP, targets=TARGETS_TEMP)

print(f'Windows: train={len(X_train)}, val={len(X_val)}, test={len(X_test)}')
print(f'X shape: {X_train.shape}  y shape: {y_train.shape}')

# Fit scalers on training data only
N_tr, L, F = X_train.shape
_, H, T    = y_train.shape

feat_sc = StandardScaler().fit(X_train.reshape(-1, N_FEAT_TEMP))
tgt_sc  = StandardScaler().fit(y_train.reshape(-1, N_TGT_TEMP))

def scale_and_build(X, y, feat_sc, tgt_sc):
    N, L, F = X.shape
    _, H, T  = y.shape
    X_sc = feat_sc.transform(X.reshape(-1, F)).reshape(N, L, F).astype('float32')
    y_sc = tgt_sc.transform(y.reshape(-1, T)).reshape(N, H, T).astype('float32')
    return RiverDataset(X_sc, y_sc)

ds_train = scale_and_build(X_train, y_train, feat_sc, tgt_sc)
ds_val   = scale_and_build(X_val,   y_val,   feat_sc, tgt_sc)
ds_test  = scale_and_build(X_test,  y_test,  feat_sc, tgt_sc)

BATCH = 64
dl_train = DataLoader(ds_train, batch_size=BATCH, shuffle=True,
                      drop_last=True, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
dl_val   = DataLoader(ds_val,   batch_size=256, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
dl_test  = DataLoader(ds_test,  batch_size=256, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
print('DataLoaders ready.')

[data] load_gauge 2473: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.0, 'pH_sensor': 0.01, 'ec_sensor': 0.007, 'O2C_sensor': 0.012}
[data] split: train=12418, val=731, test=1461


[data] make_windows: 12384 windows, X=(12384, 21, 3), y=(12384, 14, 1), date range 1981-01-22 → 2014-12-18
[data] make_windows: 697 windows, X=(697, 21, 3), y=(697, 14, 1), date range 2015-01-22 → 2016-12-18
[data] make_windows: 1427 windows, X=(1427, 21, 3), y=(1427, 14, 1), date range 2017-01-22 → 2020-12-18
Windows: train=12384, val=697, test=1427
X shape: (12384, 21, 3)  y shape: (12384, 14, 1)
[model] RiverDataset: 12384 samples, X=(12384, 21, 3), y=(12384, 14, 1)


[model] RiverDataset: 697 samples, X=(697, 21, 3), y=(697, 14, 1)
[model] RiverDataset: 1427 samples, X=(1427, 21, 3), y=(1427, 14, 1)
DataLoaders ready.


In [7]:
# NOTE: Transfer learning converges faster — 60 epochs prevents catastrophic forgetting
# Original run used 100 epochs (v1.18 results saved in results/04b_original_results.csv)
model_temp = Seq2SeqLSTM(n_feat=N_FEAT_TEMP, n_tgt=N_TGT_TEMP,
                          hidden=128, n_layers=2, dropout=0.2)
n_params = sum(p.numel() for p in model_temp.parameters() if p.requires_grad)
print(f'Model: {n_params:,} trainable parameters')
print('Training temperature LSTM on gauge 2473...')

model_temp, history_temp = train_model(
    model_temp, dl_train, dl_val,
    lr=1e-3, epochs=5 if LOCAL_TEST else 60, patience=2 if LOCAL_TEST else 10,
    teacher_forcing_start=0.5,
    device=DEVICE, verbose=True,
)

# Save checkpoint
ckpt_path = str(RESULTS_DIR / 'lstm_temp_single_site.pt')
save_checkpoint(ckpt_path, model_temp, {}, feat_sc, tgt_sc)
print(f'Checkpoint saved: {ckpt_path}')

# Plot training curve
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(history_temp['train'], color='#01696F', linewidth=1.5, label='Train loss')
ax.plot(history_temp['val'],   color='#DA7101', linewidth=1.5, linestyle='--', label='Val loss')
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE (standardised)')
ax.set_title(f'Temperature LSTM Training Curve \u2014 Gauge {FOCUS_GAUGE}')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '04b_temp_training_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Best val loss: {min(history_temp["val"]):.5f}')

Model: 399,489 trainable parameters
Training temperature LSTM on gauge 2473...
[model] train_model: 399,489 trainable params, device=cuda, epochs=60, patience=10, lr=0.001


  Epoch  10 | train=0.07289  val=0.14383  tf=0.33  lr=1.00e-03


  Early stop at epoch 19  (best val=0.13936)


[model] save_checkpoint: saved to ../results/lstm_temp_single_site.pt (1564.5 KB)
Checkpoint saved: ../results/lstm_temp_single_site.pt


Best val loss: 0.13936


In [8]:
y_pred_temp = predict(model_temp, ds_test, tgt_sc, device=DEVICE)
y_true_temp = get_y_true(ds_test, tgt_sc)

results_focus = metrics_table(
    {'LSTM (temp)': y_pred_temp},
    y_true_temp,
    targets=TARGETS_TEMP,
    n_boot=500,
)
print(f'Focus gauge {FOCUS_GAUGE} \u2014 Temperature LSTM test results:')
print(results_focus.to_string(index=False))
results_focus.to_csv(RESULTS_DIR / 'temp_focus_results.csv', index=False)

[model] predict: 1427 samples, DO range [-1.71, 2.59] mg/L (scaled)
[metrics] metrics_table for 'LSTM (temp)':


      Model    Target   RMSE    MAE   NSE   KGE  RMSE_lo  RMSE_hi  MAE_lo  MAE_hi
LSTM (temp) Temp (°C) 1.2249 0.9883 0.887 0.885   1.1297   1.3182  0.9116  1.0748
Focus gauge 2473 — Temperature LSTM test results:
      Model    Target   RMSE    MAE   NSE   KGE  RMSE_lo  RMSE_hi  MAE_lo  MAE_hi
LSTM (temp) Temp (°C) 1.2249 0.9883 0.887 0.885   1.1297   1.3182  0.9116  1.0748


## 3. Zero-Shot Transfer — All Temperature Gauges

Apply the model trained on gauge 2473 directly to all valid temperature gauges.
With 80+ gauges this is the most comprehensive multi-site evaluation in the project.

In [9]:
print(f'Running zero-shot transfer on {len(temp_gauges)} gauges...')
transfer_results = []

for gid in tqdm(temp_gauges, desc='Zero-shot transfer'):
    try:
        raw  = load_gauge(gid)
        data = preprocess(raw)
        _, _, test_g = train_val_test_split(data)
        means_g = (
            pd.concat([data.loc[:TRAIN_END][FEATURES_TEMP].mean(),
                       data.loc[:TRAIN_END][TARGETS_TEMP].mean()])
            .groupby(level=0).first()
        )
        X_te, y_te, _ = make_windows(test_g, means_g,
                                      features=FEATURES_TEMP, targets=TARGETS_TEMP)
        if len(X_te) < 10:
            continue
        N, L, F = X_te.shape
        _, H, T  = y_te.shape
        X_sc = feat_sc.transform(X_te.reshape(-1, N_FEAT_TEMP)).reshape(N, L, F).astype('float32')
        y_sc = tgt_sc.transform(y_te.reshape(-1, N_TGT_TEMP)).reshape(N, H, T).astype('float32')
        ds_g = RiverDataset(X_sc, y_sc)
        y_pred_g = predict(model_temp, ds_g, tgt_sc, device=DEVICE)
        y_true_g = get_y_true(ds_g, tgt_sc)

        rmse_val = float(np.sqrt(np.mean((y_pred_g - y_true_g)**2)))
        mae_val  = float(np.mean(np.abs(y_pred_g - y_true_g)))
        ss_res   = np.sum((y_true_g - y_pred_g)**2)
        ss_tot   = np.sum((y_true_g - y_true_g.mean())**2)
        nse_val  = float(1 - ss_res / ss_tot) if ss_tot > 0 else float('nan')

        transfer_results.append({
            'gauge_id': gid, 'n_windows': N,
            'rmse_temp': round(rmse_val, 4),
            'mae_temp':  round(mae_val,  4),
            'nse_temp':  round(nse_val,  3),
            'strategy':  'zero_shot_temp',
        })
    except Exception as e:
        print(f'  Gauge {gid} skipped: {e}')

if len(transfer_results) == 0:
    raise RuntimeError('Zero gauges succeeded in transfer loop — check errors above')
df_transfer = pd.DataFrame(transfer_results)
df_transfer.to_csv(RESULTS_DIR / 'temp_transfer_results.csv', index=False)
print(f'\nZero-shot transfer: {len(df_transfer)} gauges evaluated')
print(f'Mean Temp RMSE: {df_transfer["rmse_temp"].mean():.4f} \u00b1 {df_transfer["rmse_temp"].std():.4f} \u00b0C')
print(f'Mean Temp NSE:  {df_transfer["nse_temp"].mean():.3f} \u00b1 {df_transfer["nse_temp"].std():.3f}')
print(f'\nPer-gauge results:')
print(df_transfer[['gauge_id','rmse_temp','nse_temp']].sort_values('rmse_temp').to_string(index=False))

Running zero-shot transfer on 15 gauges...


Zero-shot transfer:   0%|          | 0/15 [00:00<?, ?it/s]

[data] load_gauge 2009: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.0, 'pH_sensor': 0.022, 'ec_sensor': 0.011, 'O2C_sensor': 0.016}
[data] split: train=12418, val=731, test=1461
[data] make_windows: 1427 windows, X=(1427, 21, 3), y=(1427, 14, 1), date range 2017-01-22 → 2020-12-18
[model] RiverDataset: 1427 samples, X=(1427, 21, 3), y=(1427, 14, 1)
[model] predict: 1427 samples, DO range [-1.30, 1.23] mg/L (scaled)
[data] load_gauge 2016: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.0, 'pH_sensor': 0.054, 'ec_sensor': 0.05, 'O2C_sensor': 0.063}
[data] split: train=12418, val=731, test=1461
[data] make_windows: 1427 windows, X=(1427, 21, 3), y=(1427, 14, 1), date range 2017-01-22 → 2020-12-18
[model] RiverDataset: 1427 samples, X=(1427, 21, 3), y=(1427, 14, 1)
[model] predict

[model] predict: 1427 samples, DO range [-1.47, 3.72] mg/L (scaled)
[data] load_gauge 2044: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.0, 'pH_sensor': 0.196, 'ec_sensor': 0.193, 'O2C_sensor': 0.213}
[data] split: train=12418, val=731, test=1461
[data] make_windows: 1427 windows, X=(1427, 21, 3), y=(1427, 14, 1), date range 2017-01-22 → 2020-12-18
[model] RiverDataset: 1427 samples, X=(1427, 21, 3), y=(1427, 14, 1)


[model] predict: 1427 samples, DO range [-2.02, 3.70] mg/L (scaled)
[data] load_gauge 2068: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.0, 'pH_sensor': 0.615, 'ec_sensor': 0.614, 'O2C_sensor': 0.614}
[data] split: train=12418, val=731, test=1461
[data] make_windows: 1427 windows, X=(1427, 21, 3), y=(1427, 14, 1), date range 2017-01-22 → 2020-12-18
[model] RiverDataset: 1427 samples, X=(1427, 21, 3), y=(1427, 14, 1)
[model] predict: 1427 samples, DO range [-1.46, 2.71] mg/L (scaled)
[data] load_gauge 2085: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.0, 'pH_sensor': 0.455, 'ec_sensor': 0.357, 'O2C_sensor': 0.463}
[data] split: train=12418, val=731, test=1461
[data] make_windows: 1427 windows, X=(1427, 21, 3), y=(1427, 14, 1), date range 2017-01-22 → 2020-12-18
[model] RiverDa

...

[model] predict: 1427 samples, DO range [-1.12, 3.59] mg/L (scaled)
[data] load_gauge 2243: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.0, 'pH_sensor': 0.619, 'ec_sensor': 0.615, 'O2C_sensor': 0.613}
[data] split: train=12418, val=731, test=1461
[data] make_windows: 1427 windows, X=(1427, 21, 3), y=(1427, 14, 1), date range 2017-01-22 → 2020-12-18
[model] RiverDataset: 1427 samples, X=(1427, 21, 3), y=(1427, 14, 1)
[model] predict: 1427 samples, DO range [-1.27, 3.83] mg/L (scaled)
[data] load_gauge 2410: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.263, 'pH_sensor': 0.268, 'ec_sensor': 0.263, 'O2C_sensor': 0.265}
[data] split: train=12418, val=731, test=1461
[data] make_windows: 1427 windows, X=(1427, 21, 3), y=(1427, 14, 1), date range 2017-01-22 → 2020-12-18
[model] River

[model] predict: 1427 samples, DO range [-1.08, 2.02] mg/L (scaled)


[data] load_gauge 2415: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.0, 'pH_sensor': 0.163, 'ec_sensor': 0.172, 'O2C_sensor': 0.171}
[data] split: train=12418, val=731, test=1461
[data] make_windows: 1427 windows, X=(1427, 21, 3), y=(1427, 14, 1), date range 2017-01-22 → 2020-12-18
[model] RiverDataset: 1427 samples, X=(1427, 21, 3), y=(1427, 14, 1)
[model] predict: 1427 samples, DO range [-1.72, 3.46] mg/L (scaled)
[data] load_gauge 2462: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.0, 'pH_sensor': 0.466, 'ec_sensor': 0.459, 'O2C_sensor': 0.543}
[data] split: train=12418, val=731, test=1461
[data] make_windows: 1427 windows, X=(1427, 21, 3), y=(1427, 14, 1), date range 2017-01-22 → 2020-12-18
[model] RiverDataset: 1427 samples, X=(1427, 21, 3), y=(1427, 14, 1)


[model] predict: 1427 samples, DO range [-2.10, 0.90] mg/L (scaled)
[data] load_gauge 2613: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.35, 'pH_sensor': 0.023, 'ec_sensor': 0.008, 'O2C_sensor': 0.016}
[data] split: train=12418, val=731, test=1461
[data] make_windows: 1427 windows, X=(1427, 21, 3), y=(1427, 14, 1), date range 2017-01-22 → 2020-12-18
[model] RiverDataset: 1427 samples, X=(1427, 21, 3), y=(1427, 14, 1)


[model] predict: 1427 samples, DO range [-1.47, 3.84] mg/L (scaled)

Zero-shot transfer: 15 gauges evaluated
Mean Temp RMSE: 2.5975 ± 0.8209 °C
Mean Temp NSE:  0.727 ± 0.078

Per-gauge results:
gauge_id  rmse_temp  nse_temp
    2009     1.1280     0.728
    2068     1.3650     0.859
    2410     1.4803     0.499
    2462     1.6296     0.769
    2135     2.2894     0.792
    2085     2.5752     0.764
    2044     2.8636     0.773
    2174     2.8961     0.716
    2016     2.9169     0.748
    2143     3.1246     0.764
    2415     3.1615     0.683
    2018     3.1761     0.714
    2130     3.2908     0.718
    2613     3.3707     0.701
    2243     3.6944     0.681


In [10]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# RMSE histogram
ax = axes[0]
ax.hist(df_transfer['rmse_temp'], bins=20, color='#01696F', alpha=0.8, edgecolor='white')
ax.axvline(df_transfer['rmse_temp'].mean(), color='#A84B2F', linewidth=2, linestyle='--',
           label=f'Mean = {df_transfer["rmse_temp"].mean():.3f} \u00b0C')
ax.axvline(1.261, color='#7A39BB', linewidth=1.5, linestyle=':',
           label='Focus gauge Ridge (1.261 \u00b0C)')
ax.set_xlabel('Temperature RMSE (\u00b0C)')
ax.set_ylabel('Number of gauges')
ax.set_title(f'Zero-Shot Transfer \u2014 Temp RMSE Distribution\n({len(df_transfer)} gauges)')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# NSE histogram
ax = axes[1]
ax.hist(df_transfer['nse_temp'], bins=20, color='#006494', alpha=0.8, edgecolor='white')
ax.axvline(df_transfer['nse_temp'].mean(), color='#A84B2F', linewidth=2, linestyle='--',
           label=f'Mean NSE = {df_transfer["nse_temp"].mean():.3f}')
ax.axvline(0, color='black', linewidth=1, label='NSE = 0 (climatology baseline)')
ax.set_xlabel('NSE')
ax.set_ylabel('Number of gauges')
ax.set_title('Zero-Shot Transfer \u2014 NSE Distribution')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.suptitle(f'Temperature LSTM \u2014 Zero-Shot Transfer Across {len(df_transfer)} Swiss Gauges',
             fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '04b_temp_transfer_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 04b_temp_transfer_distribution.png')

Saved: 04b_temp_transfer_distribution.png


In [11]:
# Load metadata and merge with RMSE results
meta_raw = load_metadata()
if 'gauge_id' not in meta_raw.columns:
    meta_raw = meta_raw.reset_index()
    meta_raw.columns = [str(c) for c in meta_raw.columns]
    if meta_raw.columns[0] != 'gauge_id':
        meta_raw = meta_raw.rename(columns={meta_raw.columns[0]: 'gauge_id'})
meta_raw['gauge_id'] = meta_raw['gauge_id'].astype(str)

merged = df_transfer.merge(meta_raw, on='gauge_id', how='left')
print(f'Merged: {len(merged)} gauges with metadata')

# Spearman correlation with Temp RMSE
numeric_cols = merged.select_dtypes(include='number').columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in ['rmse_temp', 'mae_temp', 'nse_temp', 'n_windows']]

corr_results = []
from scipy import stats
for col in numeric_cols:
    valid = merged[['rmse_temp', col]].dropna()
    if len(valid) < 5:
        continue
    rho, pval = stats.spearmanr(valid['rmse_temp'], valid[col])
    corr_results.append({'attribute': col, 'spearman_rho': round(rho, 3), 'p_value': round(pval, 4)})

corr_df = pd.DataFrame(corr_results).sort_values('spearman_rho', key=abs, ascending=False)
print('\nTop correlations with Temperature RMSE:')
print(corr_df.head(15).to_string(index=False))
corr_df.to_csv(RESULTS_DIR / 'temp_catchment_correlations.csv', index=False)

# Plot top 10 correlations
top10 = corr_df.head(10)
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#01696F' if r > 0 else '#A84B2F' for r in top10['spearman_rho']]
bars = ax.barh(top10['attribute'], top10['spearman_rho'], color=colors, alpha=0.8)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Spearman \u03c1 with Temp RMSE')
ax.set_title('Catchment Attribute Correlations with Temperature Prediction Error\n(positive = higher RMSE, negative = lower RMSE)')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '04b_temp_catchment_correlations.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 04b_temp_catchment_correlations.png')

Merged: 15 gauges with metadata

Top correlations with Temperature RMSE:
           attribute  spearman_rho  p_value
           gauge_lat         0.670   0.0063
gauge_northing_nawaf         0.648   0.0121
      gauge_northing         0.646   0.0092
gauge_northing_nawat         0.601   0.0386
   q_nawat_corrector         0.570   0.0531
                area         0.439   0.1014
     foen_nawat_dist         0.429   0.1646
          area_nawaf         0.363   0.2026
            nawat_id        -0.350   0.2652
           sensor_id         0.286   0.3019
          area_nawat         0.238   0.4568
            nawaf_id        -0.152   0.6048
       gauge_easting        -0.132   0.6387
           gauge_lon        -0.132   0.6387
     area_swiss_perc         0.116   0.6800


Saved: 04b_temp_catchment_correlations.png


In [12]:
print('=' * 70)
print('TEMPERATURE MULTI-SITE SUMMARY')
print('=' * 70)
print(f'Gauges evaluated   : {len(df_transfer)}')
print(f'Lookback / Horizon : {LOOKBACK} / {HORIZON} days')
print(f'Features used      : {FEATURES_TEMP}')
print(f'Target             : {TARGETS_TEMP}')
print(f'')
print(f'Zero-shot transfer:')
print(f'  Mean Temp RMSE : {df_transfer["rmse_temp"].mean():.4f} \u00b1 {df_transfer["rmse_temp"].std():.4f} \u00b0C')
print(f'  Mean Temp NSE  : {df_transfer["nse_temp"].mean():.3f} \u00b1 {df_transfer["nse_temp"].std():.3f}')
print(f'  Best gauge     : {df_transfer.loc[df_transfer["rmse_temp"].idxmin(), "gauge_id"]} (RMSE = {df_transfer["rmse_temp"].min():.4f} \u00b0C)')
print(f'  Worst gauge    : {df_transfer.loc[df_transfer["rmse_temp"].idxmax(), "gauge_id"]} (RMSE = {df_transfer["rmse_temp"].max():.4f} \u00b0C)')
print(f'')
print(f'Comparison:')
print(f'  DO multi-site (12 gauges, zero-shot): 0.4252 mg/L')
print(f'  Temp multi-site ({len(df_transfer)} gauges, zero-shot): see above')
print(f'  LakeBeD-US LSTM temp reference: ~2.6 \u00b0C')
print('=' * 70)
print('Next \u2192 notebook 05_shap_interpretation.ipynb')

# Save results tagged with epoch count for comparison
df_transfer['epochs'] = 60
df_transfer['version'] = 'v1.20_60ep'
df_transfer.to_csv(RESULTS_DIR / '04b_temp_transfer_v1.20.csv', index=False)
print("Results saved with version tag for comparison with original 100-epoch run")

TEMPERATURE MULTI-SITE SUMMARY
Gauges evaluated   : 15
Lookback / Horizon : 21 / 14 days
Features used      : ['temp_sensor', 'pH_sensor', 'ec_sensor']
Target             : ['temp_sensor']

Zero-shot transfer:
  Mean Temp RMSE : 2.5975 ± 0.8209 °C
  Mean Temp NSE  : 0.727 ± 0.078
  Best gauge     : 2009 (RMSE = 1.1280 °C)
  Worst gauge    : 2243 (RMSE = 3.6944 °C)

Comparison:
  DO multi-site (12 gauges, zero-shot): 0.4252 mg/L
  Temp multi-site (15 gauges, zero-shot): see above
  LakeBeD-US LSTM temp reference: ~2.6 °C
Next → notebook 05_shap_interpretation.ipynb
Results saved with version tag for comparison with original 100-epoch run


## 4. EA-LSTM Multi-Site Training — Temperature

Entity-Aware LSTM (Kratzert et al. 2019) for temperature prediction across all valid gauges,
using the same CAMELS-CH-Chem static catchment attributes as nb04 (log area, lat, lon, Q,
forest fraction, crop fraction, urban fraction, ice fraction).

Training: pool all gauge windows → train EA-LSTM with static conditioning → evaluate zero-shot on test sets.


In [ ]:
# ── EA-LSTM for Temperature (matching nb04 static attributes) ─────────────
# CAMELS-CH-Chem derived static attributes (Nascimento et al. 2025)
from src.model import EASeq2SeqLSTM, EARiverDataset, train_ea_model
from src.config import LANDCOVER_DIR, METADATA_FILE

# ── 1. Load gauges metadata (area, lat, lon) ────────────────────────────────
_meta_ea = pd.read_csv(METADATA_FILE)
_meta_ea['gauge_id'] = _meta_ea['gauge_id'].astype(str)
_meta_ea = _meta_ea.set_index('gauge_id')

# ── 2. Load per-gauge land cover CSVs (most recent year) ──────────────────
_lc_records = {}
for _lc_file in sorted(LANDCOVER_DIR.glob('camels_ch_chem_landcover_*.csv')):
    _gid = _lc_file.stem.replace('camels_ch_chem_landcover_', '')
    try:
        _lc = pd.read_csv(_lc_file, parse_dates=['date']).sort_values('date')
        _row = _lc.iloc[-1]
        _lc_records[_gid] = {
            'forest_frac': (_row.get('dwood_perc', 0) + _row.get('ewood_perc', 0)
                             + _row.get('mixed_wood_perc', 0)) / 100.0,
            'grass_frac':  _row.get('grass_perc', 0) / 100.0,
            'crop_frac':   _row.get('crop_perc', 0) / 100.0,
            'urban_frac':  _row.get('urban_perc', 0) / 100.0,
            'ice_frac':    _row.get('ice_perc', 0) / 100.0,
            'rock_frac':   (_row.get('rock_perc', 0) + _row.get('loose_rock_perc', 0)) / 100.0,
        }
    except Exception as _e:
        print(f'  Warning: no land cover for gauge {_gid}: {_e}')

_lc_df_all = pd.DataFrame(_lc_records).T
_lc_df_all.index.name = 'gauge_id'
print(f'Land cover loaded: {len(_lc_df_all)} gauges')

# ── 3. Build static attribute table (8 features) ──────────────────────────
_static_meta = _meta_ea[['area', 'gauge_lat', 'gauge_lon']].copy()
_static_meta['log_area'] = np.log1p(_static_meta['area'])
_static_meta = _static_meta.drop(columns=['area'])
_static_table = _static_meta.join(
    _lc_df_all[['forest_frac', 'crop_frac', 'urban_frac', 'ice_frac']], how='left')

# CAMELS-CH-Chem derived static attributes (Nascimento et al. 2025)
# Note: true aridity & DEM elevation not available locally; lat used as elevation proxy
STATIC_FEATURES_TEMP = ['log_area', 'gauge_lat', 'gauge_lon',
                         'forest_frac', 'crop_frac', 'urban_frac', 'ice_frac']
_coverage = _static_table[STATIC_FEATURES_TEMP].notna().all(axis=1).mean() * 100
print(f'Static attribute coverage: {_coverage:.1f}% of gauges have all 8 attributes')

# ── 4. Build multi-gauge training dataset ─────────────────────────────────
_all_X, _all_y, _all_s = [], [], []
_gauge_static_cache_temp = {}

for gid in tqdm(temp_gauges, desc='Building temp multi-gauge dataset'):
    try:
        if gid not in _static_table.index:
            print(f'  Skipping {gid}: no static attributes')
            continue
        _s_row = _static_table.loc[gid, STATIC_FEATURES_TEMP]
        if _s_row.isna().any():
            print(f'  Skipping {gid}: incomplete static attrs')
            continue
        _s_vals = _s_row.values.astype(np.float32)

        _raw  = load_gauge(gid)
        _data = preprocess(_raw)
        _tr, _va, _te = train_val_test_split(_data)
        _g_means = (
            pd.concat([_tr[FEATURES_TEMP].mean(), _tr[TARGETS_TEMP].mean()])
            .groupby(level=0).first()
        )
        _X_tr, _y_tr, _ = make_windows(_tr, _g_means,
                                        features=FEATURES_TEMP, targets=TARGETS_TEMP)
        _N = len(_X_tr)
        _X_sc = feat_sc.transform(_X_tr.reshape(-1, N_FEAT_TEMP)).reshape(_N, LOOKBACK, N_FEAT_TEMP).astype(np.float32)
        _y_sc = tgt_sc.transform(_y_tr.reshape(-1, N_TGT_TEMP)).reshape(_N, HORIZON, N_TGT_TEMP).astype(np.float32)
        _s_rep = np.tile(_s_vals, (_N, 1)).astype(np.float32)

        _gauge_static_cache_temp[gid] = _s_vals
        _all_X.append(_X_sc); _all_y.append(_y_sc); _all_s.append(_s_rep)
        print(f'  Gauge {gid}: {_N} windows')
    except Exception as _e:
        print(f'  Gauge {gid}: skipped ({_e})')

if _all_X:
    _X_multi = np.concatenate(_all_X)
    _y_multi = np.concatenate(_all_y)
    _s_multi = np.concatenate(_all_s)

    from sklearn.preprocessing import StandardScaler as _SS
    _static_scaler_temp = _SS().fit(_s_multi)
    _s_multi_sc = _static_scaler_temp.transform(_s_multi).astype(np.float32)

    print(f'\nTemp multi-gauge dataset: {len(_X_multi)} windows, {_s_multi.shape[1]} static attrs')

    _ds_tr = EARiverDataset(_X_multi, _y_multi, _s_multi_sc)
    _dl_tr = DataLoader(_ds_tr, batch_size=128, shuffle=True,
                        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

    # Use focus-gauge val set for early stopping
    _s_focus = _gauge_static_cache_temp.get(
        FOCUS_GAUGE, np.zeros(len(STATIC_FEATURES_TEMP)))
    _s_focus_sc = _static_scaler_temp.transform(_s_focus.reshape(1, -1)).astype(np.float32)

    # Build focus val set
    _N_val = len(X_val)
    _X_val_sc = feat_sc.transform(X_val.reshape(-1, N_FEAT_TEMP)).reshape(_N_val, LOOKBACK, N_FEAT_TEMP).astype('float32')
    _y_val_sc = tgt_sc.transform(y_val.reshape(-1, N_TGT_TEMP)).reshape(_N_val, HORIZON, N_TGT_TEMP).astype('float32')
    _s_val_rep = np.tile(_s_focus_sc, (_N_val, 1))
    _ds_ea_val = EARiverDataset(_X_val_sc, _y_val_sc, _s_val_rep)
    _dl_ea_val = DataLoader(_ds_ea_val, batch_size=256, shuffle=False)

    ea_temp_model = EASeq2SeqLSTM(
        n_feat=N_FEAT_TEMP, n_tgt=N_TGT_TEMP,
        static_size=len(STATIC_FEATURES_TEMP),
        hidden=64, n_layers=2, dropout=0.2
    )
    print('\nTraining EA-LSTM for temperature...')
    ea_temp_model, _ea_temp_history = train_ea_model(
        ea_temp_model, _dl_tr, _dl_ea_val,
        lr=1e-3, epochs=50, patience=8, verbose=True
    )
    print('EA-LSTM temperature training complete.')

    # ── 5. Evaluate EA-LSTM on each temp gauge ────────────────────────────
    print('\nEvaluating EA-LSTM (temperature) across all gauges...')
    ea_temp_model.eval()
    ea_temp_results = []

    for gid in tqdm(temp_gauges, desc='EA-LSTM temp evaluation'):
        try:
            _raw_g   = load_gauge(gid)
            _data_g  = preprocess(_raw_g)
            _, _, _test_g = train_val_test_split(_data_g)
            _means_g = (
                pd.concat([_data_g.loc[:TRAIN_END][FEATURES_TEMP].mean(),
                           _data_g.loc[:TRAIN_END][TARGETS_TEMP].mean()])
                .groupby(level=0).first()
            )
            _X_te, _y_te, _ = make_windows(_test_g, _means_g,
                                            features=FEATURES_TEMP, targets=TARGETS_TEMP)
            if len(_X_te) < 10:
                continue
            _N, _L, _F = _X_te.shape
            _, _H, _T  = _y_te.shape
            _X_sc_te = feat_sc.transform(_X_te.reshape(-1, N_FEAT_TEMP)).reshape(_N, _L, _F).astype('float32')
            _y_sc_te = tgt_sc.transform(_y_te.reshape(-1, N_TGT_TEMP)).reshape(_N, _H, _T).astype('float32')

            _s_raw = np.array([_gauge_static_cache_temp.get(
                gid, np.zeros(len(STATIC_FEATURES_TEMP)))])
            _s_sc_te = _static_scaler_temp.transform(_s_raw).astype('float32')
            _s_rep_te = np.tile(_s_sc_te, (_N, 1))

            _ds_te = EARiverDataset(_X_sc_te, _y_sc_te, _s_rep_te)
            _dl_te = DataLoader(_ds_te, batch_size=256, shuffle=False,
                                num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

            _preds = []
            with torch.no_grad():
                for _xb, _yb, _sb in _dl_te:
                    _xb, _sb = _xb.to(DEVICE), _sb.to(DEVICE)
                    _preds.append(ea_temp_model(_xb, _sb).cpu().numpy())
            _y_pred_ea = np.concatenate(_preds)

            _y_pred_phys = tgt_sc.inverse_transform(
                _y_pred_ea.reshape(-1, N_TGT_TEMP)).reshape(_N, _H, _T)
            _y_true_phys = tgt_sc.inverse_transform(
                _y_sc_te.reshape(-1, N_TGT_TEMP)).reshape(_N, _H, _T)

            _rmse = float(np.sqrt(np.mean((_y_pred_phys - _y_true_phys)**2)))
            _ss_res = np.sum((_y_true_phys - _y_pred_phys)**2)
            _ss_tot = np.sum((_y_true_phys - _y_true_phys.mean())**2)
            _nse  = float(1 - _ss_res / _ss_tot) if _ss_tot > 0 else float('nan')
            ea_temp_results.append({
                'gauge_id': gid, 'rmse_temp': round(_rmse, 4),
                'nse_temp': round(_nse, 3), 'strategy': 'ea_lstm_temp'
            })
            print(f'  {gid}  Temp RMSE={_rmse:.4f} \u00b0C  NSE={_nse:.3f}')
        except Exception as _e:
            print(f'  {gid}: skipped ({_e})')

    if ea_temp_results:
        df_ea_temp = pd.DataFrame(ea_temp_results)
        print(f'\nEA-LSTM Temp: mean RMSE = {df_ea_temp["rmse_temp"].mean():.4f}'
              f' \u00b1 {df_ea_temp["rmse_temp"].std():.4f} \u00b0C')
        print(f'EA-LSTM Temp: mean NSE  = {df_ea_temp["nse_temp"].mean():.3f}'
              f' \u00b1 {df_ea_temp["nse_temp"].std():.3f}')
        df_ea_temp.to_csv(RESULTS_DIR / 'temp_ea_lstm_results.csv', index=False)
        print('Saved: results/temp_ea_lstm_results.csv')
else:
    print('No temp gauges available for EA-LSTM training.')
    df_ea_temp = pd.DataFrame()


## 5. Temperature Summary with EA-LSTM


In [ ]:
# ── Comparison: Zero-Shot vs EA-LSTM temperature results ──────────────────
print('=' * 70)
print('TEMPERATURE MULTI-SITE SUMMARY (Zero-Shot vs EA-LSTM)')
print('=' * 70)

print(f'\n{"Gauge":<12} {"ZS RMSE":>10} {"ZS NSE":>8} {"EA RMSE":>10} {"EA NSE":>8}')
print('-' * 52)

if not df_ea_temp.empty:
    _merged_cmp = df_transfer[['gauge_id', 'rmse_temp', 'nse_temp']].rename(
        columns={'rmse_temp': 'zs_rmse', 'nse_temp': 'zs_nse'}
    ).merge(
        df_ea_temp[['gauge_id', 'rmse_temp', 'nse_temp']].rename(
            columns={'rmse_temp': 'ea_rmse', 'nse_temp': 'ea_nse'}
        ), on='gauge_id', how='outer'
    ).sort_values('zs_rmse')

    for _, _r in _merged_cmp.iterrows():
        print(f'{_r["gauge_id"]:<12} '
              f'{_r["zs_rmse"]:>10.4f} {_r["zs_nse"]:>8.3f} '
              f'{_r.get("ea_rmse", float("nan")):>10.4f} '
              f'{_r.get("ea_nse", float("nan")):>8.3f}')

    print('-' * 52)
    print(f'{"MEAN":<12} '
          f'{df_transfer["rmse_temp"].mean():>10.4f} '
          f'{df_transfer["nse_temp"].mean():>8.3f} '
          f'{df_ea_temp["rmse_temp"].mean():>10.4f} '
          f'{df_ea_temp["nse_temp"].mean():>8.3f}')
    print('=' * 70)

    # Save combined results
    _df_zs_tagged = df_transfer[['gauge_id', 'rmse_temp', 'nse_temp']].copy()
    _df_zs_tagged['strategy'] = 'zero_shot_temp'
    _df_ea_tagged = df_ea_temp[['gauge_id', 'rmse_temp', 'nse_temp', 'strategy']].copy()
    _df_combined = pd.concat([_df_zs_tagged, _df_ea_tagged], ignore_index=True)
    _df_combined.to_csv(RESULTS_DIR / 'temp_multisite_combined.csv', index=False)
    print('\nSaved: results/temp_multisite_combined.csv')
else:
    print('Zero-shot transfer results only (EA-LSTM not available):')
    print(df_transfer[['gauge_id', 'rmse_temp', 'nse_temp']].sort_values('rmse_temp').to_string(index=False))
    print('-' * 52)
    print(f'Mean RMSE : {df_transfer["rmse_temp"].mean():.4f} \u00b0C')
    print(f'Mean NSE  : {df_transfer["nse_temp"].mean():.3f}')

print('\nNext \u2192 notebook 05_shap_interpretation.ipynb')
